In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = "......"
model = ChatGroq(model = "qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A2396A41A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A2396A4EC0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

Pydantic

In [2]:
from pydantic import BaseModel, Field 

class Movie(BaseModel):
    title:str=Field(description="The title of the movie.")
    year:int=Field(description="The year of the movie was released.")
    director:str=Field(description="The director of the movie.")
    rating:float=Field(description="The movie rating out of 10.")

In [ ]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A2396A41A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A2396A4EC0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie.', 'type': 'string'}, 'year': {'description': 'The year of the movie was released.', 'type': 'integer'}, 'director': {'description': 'The director of the movie.', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10.', 'type': 'number'}}, 'required': ['title', 'yea

In [4]:
model.invoke("Provide the detail about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Directed by Christopher Nolan, right? It came out in 2010. The main actor is Leonardo DiCaprio. The genre is sci-fi, maybe a bit of action. The title "Inception" makes me think it\'s about planting an idea, maybe in someone\'s mind. I remember something about dreams within dreams, like multiple layers. There\'s a concept called "shared dreaming" where people enter each other\'s dreams. \n\nThe plot probably involves a team of people using this technology to plant an idea into someone\'s subconscious. The main character, Dom Cobb (DiCaprio), is a thief who steals information by entering people\'s dreams. But in this case, he\'s trying to do the opposite—plant an idea. He\'s been offered a chance to erase his criminal past by performing the reverse, which is the inception. The team includes different experts: an architect, a forger, a dreamshare specia

In [5]:
model_with_structure.invoke("Provide the detail about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

Messaged output alongside Parased structure

In [6]:
from pydantic import BaseModel, Field 

class Movie(BaseModel):
    title:str=Field(description="The title of the movie.")
    year:int=Field(description="The year of the movie was released.")
    director:str=Field(description="The director of the movie.")
    rating:float=Field(description="The movie rating out of 10.")

model_with_structure=model.with_structured_output(Movie,include_raw=True)    

response=model_with_structure.invoke("Provide the detail about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception. \n\nFirst, the title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb does have it at 8.8. \n\nSo all the required parameters are present. I should structure the tool call with these details. Make sure the JSON is correctly formatted with the right data types: title as a string, year as an integer, director as a string, and rating as a number. No missing fields, so the function call should work.\n', 'tool_calls': [{'id': '6adj8sc21', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': '

Nested Structure

In [17]:

from pydantic import BaseModel, Field 
class Actor (BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float| None = Field(None,description="Budget in million USD ")

model_with_structure=model.with_structured_output( MovieDetails)    

response=model_with_structure.invoke("Provide the detail about the movie Inception")
response      



MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Mal')], genres=['Science Fiction', 'Action', 'Adventure'], budget=160.0)

TypedDict

In [18]:
from typing_extensions import TypedDict, Annotated

In [19]:
class MovieDict(TypedDict):
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[str,...,"The year of the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[str,...,"The rating of the movie out of 10"]

In [20]:
model_withtypedict  =model.with_structured_output( MovieDict)    

response=model_withtypedict.invoke("Provide the detail about the movie avengers")
response  

{'director': 'Joss Whedon',
 'rating': '8.0',
 'title': 'Avengers',
 'year': '2012'}

In [22]:
class Actor (TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float| None = Field(None,description="Budget in million USD ")

model_with_structure=model.with_structured_output( MovieDetails)    

response=model_with_structure.invoke("Provide the detail about the movie Avengers")
response 

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}